In [ ]:
import transformers
import torch
import random
from datasets import load_dataset
import requests

#question = "Mike Barnett negotiated many contracts including which player that went on to become general manager of CSKA Moscow of the Kontinental Hockey League?"

# Model ID and device setup
model_id = "PeterJinGo/SearchR1-nq_hotpotqa_train-qwen2.5-7b-em-ppo"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# #question = question.strip()
# if question[-1] != '?':
#     question += '?'
curr_eos = [151645, 151643] # for Qwen2.5 series models
curr_search_template = '\n\n{output_text}<information>{search_results}</information>\n\n'

In [2]:
# Initialize the tokenizer and model
tokenizer = transformers.AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = transformers.AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)

Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

In [4]:
import numpy as np
from collections import namedtuple
from typing import Tuple, Dict, List, Any, Union
import torch.utils
from nltk.probability import gt_demo
from torch.utils.data import Dataset

import sys
import os
sys.path.append(os.path.abspath('/home/a.anokhin/Judge/Q-RAG-pqn'))

# from rl.jax_text_env import TextEnv, TextMemory, TextMemoryItem
from envs.text_env import TextEnv, TextMemory, TextMemoryItem
from transformers import PreTrainedTokenizer, PreTrainedTokenizerFast

    
class SimpleEnvAdapter(Dataset):
    """
    Simple adapter that adapts datasets Babilong, HotPotQA and MUSIQUE for QAREtreievalEnv.
    This adapter doesn't tokenize or embeds text chunks.

    You can create different adapter that for example tokenize every text in a sample or
    build faiss index over text chunks.
    """

    def __init__(self, dataset, min_chunks=6): # Добавили параметр min_chunks
        super().__init__()
        
        original_dataset = dataset
        self.dataset_name = original_dataset.name()
        
        # --- НАЧАЛО ИЗМЕНЕНИЙ ---
        
        print(f"Фильтрация датасета '{self.dataset_name}'. Исходный размер: {len(original_dataset)}.")
        print(f"Удаляются сэмплы, где количество чанков (контекстных параграфов) меньше {min_chunks}.")
        
        filtered_dataset = []
        for sample in original_dataset:
            # Логика определения количества чанков должна соответствовать тому,
            # как они создаются в __getitem__. Для hotpotqa это len(sample['context']).
            num_chunks = 0
            if self.dataset_name == 'hotpotqa':
                num_chunks = len(sample.get('context', []))
            elif self.dataset_name == 'musique':
                num_chunks = len(sample.get('paragraphs', []))
            # elif self.dataset_name == 'babilong':
            #     num_chunks = len(sample.get('chunks', []))
            
            if num_chunks >= min_chunks:
                filtered_dataset.append(sample)

        self.dataset = filtered_dataset
        
        print(f"Фильтрация завершена. Новый размер датасета: {len(self.dataset)}.")

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        sample = self.dataset[index]
        question = sample["question"]
        if question.endswith("?"):
            question = question[:-1]

        sf_idx = []
        chunks_texts = []
        if self.dataset_name == 'hotpotqa':
            sp_title_set = set()
            sample_id = sample['_id']
            for sup in sample['supporting_facts']:
                sp_title_set.add(sup[0])

            for idx, (title, sentences) in enumerate(sample['context']):
                if title in sp_title_set:
                    sf_idx.append(idx)
                chunk = title + " " + " ".join(sentences)
                chunks_texts.append(chunk)

        elif self.dataset_name == 'musique':
            sample_id = sample['id']
            for i, para in enumerate(sample['paragraphs']):
                # if para['is_supporting']:
                #     sf_idx.append(i)
                chunk = para['title'] + '. ' + para['paragraph_text']
                chunks_texts.append(chunk)

            # label order
            for item_json in sample['question_decomposition']:
                sf_idx.append(item_json['paragraph_support_idx'])

        elif self.dataset_name == 'babilong':
            sample_id = index
            for i, sent in enumerate(sample['chunks']):
                chunks_texts.append(sent)

            for i in sample['references_idx']:
                sf_idx.append(i)

        return {
            'id': sample_id,
            'question': question,
            'answer': sample["answer"],
            'chunks_texts': chunks_texts,
            'sf_idx': sf_idx,
        }

In [5]:
import json
import torch
import numpy as np
from tqdm.auto import tqdm
from torch.utils.data import Dataset
import logging

logger = logging.getLogger(__name__)

class RetrievalHotPotQA(Dataset):
    def __init__(self,
                 path,
                 tokenizer,
                 length = -1,
                 min_context_len = -1,
                 max_context_len = 1e7,
                 seed = 52,
                 **kwargs
        ):
        super().__init__()
        split = kwargs.pop('split')
        if split not in ['train', 'eval', 'all',  'fullwiki_eval']:
            #'all' includes 'train' and 'eval' but not fullwiki_eval
            raise ValueError(f'unknown split for hotpotqa dataset: {split}')
        self.split = split
        logger.info(f"{type(self)} received unknown kwargs: {kwargs}")
        self.length = length
        self.min_context_len = min_context_len
        self.max_context_len = max_context_len
        self.tokenizer = tokenizer

        np.random.seed(seed)
        self._load_data(path)

    def name(self):
        return 'hotpotqa'

    def _load_data(self, path):
        self.tasks = []

        raw_tasks = []
        if self.split in ['eval', 'all']:
            with open(path + '/hotpot_dev_distractor_v1.json', 'r') as json_file:
                raw_tasks.extend(map(lambda x: (x, 'eval'), json.load(json_file)))

        if self.split in ['train', 'all']:
            with open(path + '/hotpot_train_v1.1.json', 'r') as json_file:
                raw_tasks.extend(map(lambda x: (x, 'train'), json.load(json_file)))

        if self.split == 'fullwiki_eval':
            with open(path + '/hotpot_dev_fullwiki_v1.json', 'r') as json_file:
                raw_tasks.extend(map(lambda x: (x, 'fullwiki_eval'), json.load(json_file)))


        for task, partition in tqdm(raw_tasks, "HotPotQA load"):
            context = " ".join(title + " ".join(sentences) for title, sentences in task["context"])
            context = task['question'] + context
            context_len = len(self.tokenizer(context)["input_ids"])

            if self.min_context_len <= context_len <= self.max_context_len:
                self.tasks.append(self._adapt_raw_sample(task))

        self.tasks = np.random.permutation(self.tasks)
        if self.length >= 0:
            self.tasks = self.tasks[:self.length]


    def _adapt_raw_sample(self, sample):
        """Adapt sample to unified format expected by the model"""
        return sample


    def __len__(self):
        return len(self.tasks)

    def __getitem__(self, idx):
        return self.tasks[idx]

In [4]:
import json
import torch
import numpy as np
from tqdm.auto import tqdm
from torch.utils.data import Dataset
import logging

class LocalSetMusique(Dataset):
    def __init__(self, path, tokenizer, length = -1, min_context_len = -1, max_context_len = 1e7, type = "qa", anno_type = "real", seed = 52):
        super().__init__()
        self.length = length
        self.min_context_len = min_context_len
        self.max_context_len = max_context_len
        self.type = type
        self.anno_type = anno_type
        self.tokenizer = tokenizer

        np.random.seed(seed)
        self._load_data(path)

    def name(self):
        return 'musique'

    def _load_data(self, path):
        self.tasks = []

        if self.type not in ["qa", "any"] or self.anno_type not in ["real", "any"]:
            return

        with open(path + '/musique_ans_v1.0_dev.jsonl', 'r') as json_file:
            json_list = list(json_file)
            raw_tasks = [(json.loads(json_str), "dev") for json_str in json_list]

        with open(path + '/musique_ans_v1.0_train.jsonl', 'r') as json_file:
            json_list = list(json_file)
            raw_tasks += [(json.loads(json_str), "train") for json_str in json_list]

        for task, partition in tqdm(raw_tasks, "MuSiQue load"):
            context = ""
            for text in task["paragraphs"]:
                title = text["title"]
                paragraph_text = text["paragraph_text"]
                context += f"TITLE: {title}\nTEXT: {paragraph_text}\n\n"
            context_len = len(self.tokenizer(context)["input_ids"])

            if context_len > self.max_context_len or context_len < self.min_context_len:
                continue
            self.tasks.append(Task(
                    "qa", "real", context_len, context, task["answer"], task["question"], "MuSiQue", partition,
            ))

        self.tasks = np.random.permutation(self.tasks)
        if self.length >= 0:
            self.tasks = self.tasks[:self.length]


    def __len__(self):
        return len(self.tasks)

    def __getitem__(self, idx):
        return self.tasks[idx]


class RetrievalMusique(LocalSetMusique):
    """A default version of Musique Dataset. """

    def __init__(self, path, tokenizer, length=-1, min_context_len=-1, max_context_len=1e7,
                 type="qa", anno_type="real", split='train', seed=52):
        self.split = split #possible values: 'eval', 'train', 'all'
        super().__init__(path, tokenizer, length, min_context_len, max_context_len, type, anno_type, seed)

    def _load_data(self, path):
        self.tasks = []

        if self.type not in ["qa", "any"] or self.anno_type not in ["real", "any"]:
            return

        raw_tasks = []
        if self.split in ['eval', 'all']:
            with open(path + '/musique_ans_v1.0_dev.jsonl', 'r') as json_file:
                json_list = list(json_file)
                raw_tasks.extend( [(json.loads(json_str), "dev") for json_str in json_list] )

        if self.split in ['train', 'all']:
            with open(path + '/musique_ans_v1.0_train.jsonl', 'r') as json_file:
                json_list = list(json_file)
                raw_tasks.extend( [(json.loads(json_str), "train") for json_str in json_list] )

        for sample, partition in tqdm(raw_tasks, "MuSiQue load"):
            context = ""
            for text in sample["paragraphs"]:
                title = text["title"]
                paragraph_text = text["paragraph_text"]
                context += f"TITLE: {title}\nTEXT: {paragraph_text}\n\n"
            context_len = len(self.tokenizer(context)["input_ids"])

            if self.min_context_len <= context_len <= self.max_context_len:
                self.tasks.append(self._adapt_raw_sample(sample))

        self.tasks = np.random.permutation(self.tasks)
        if self.length >= 0:
            self.tasks = self.tasks[:self.length]

    def _adapt_raw_sample(self, sample):
        """Adapt sample to unified format expected by the model"""
        return sample

In [6]:
from transformers import AutoTokenizer
seed = 42
split = 'eval'
tokenizer_for_retrieval = AutoTokenizer.from_pretrained('intfloat/e5-base-v2')
path = '/home/a.anokhin/Judge/datasets/data_sources/hotpotqa'

dataset = RetrievalHotPotQA(
    path=path, tokenizer=tokenizer_for_retrieval, length=-1,
    min_context_len=0, max_context_len=1e7,
    type='any', anno_type='any', split=split, seed=seed
)

HotPotQA load:   0%|          | 0/7405 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1044 > 512). Running this sequence through the model will result in indexing errors


In [2]:
from transformers import AutoTokenizer
seed = 42
split = 'eval'
tokenizer_for_retrieval = AutoTokenizer.from_pretrained('intfloat/e5-base-v2')
path = '/home/a.anokhin/Judge/datasets/data_sources/musique'

dataset = RetrievalMusique(
    path=path, tokenizer=tokenizer_for_retrieval, length=-1,
    min_context_len=0, max_context_len=1e7,
    type='any', anno_type='any', split=split, seed=seed
)

NameError: name 'RetrievalMusique' is not defined

In [7]:
dataset = SimpleEnvAdapter(dataset)

Фильтрация датасета 'hotpotqa'. Исходный размер: 7405.
Удаляются сэмплы, где количество чанков (контекстных параграфов) меньше 6.
Фильтрация завершена. Новый размер датасета: 7362.


In [8]:
dataset[0]

{'id': '5a7613c15542994ccc9186bf',
 'question': "VIVA Media AG changed it's name in 2004. What does their new acronym stand for",
 'answer': 'Gesellschaft mit beschränkter Haftung',
 'chunks_texts': ['Constantin Medien Constantin Medien AG (formerly EM.Entertainment and EM.TV & Merchandising AG, then EM.TV AG, and finally em.sport media ag) is a German media group, based in Ismaning near Munich, active in the area of sports, film and event marketing to medium-sized media companies.',
  'VIVA Poland VIVA Polska (earlier "VIVApolska!")  is a Polish 24h music and entertainment channel from Viacom International Media Networks Polska.  The channel was officially launched on June 10, 2000 by the German VIVA Media AG.',
  'Viva (UK and Ireland) Viva (stylised as VIVA) is a music television channel in the United Kingdom and Ireland, owned by VIVA Media and thereby Viacom International Media Networks Europe.  The channel launched on 26 October 2009, replacing TMF.',
  'Blic Blic (Cyrillic: Блиц

In [9]:
# Возьмем пример для демонстрации
sample = dataset[4] 
question = sample['question']
ground_truth_answer = sample['answer']
context_paragraphs = sample['chunks_texts']


print(f"\nВопрос: {question}")
print(f"Правильный ответ: {ground_truth_answer}")
print(f"Количество абзацев для поиска: {len(context_paragraphs)}")


Вопрос: Woman's Era and Naj are what kind of magazines
Правильный ответ: fortnightly women interest magazine
Количество абзацев для поиска: 10


In [ ]:
# from transformers import AutoTokenizer, AutoModel
# # 4. Инициализируем модель-ретривер для семантического поиска
# model_name = 'intfloat/e5-base-v2'
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModel.from_pretrained(model_name)

In [11]:
from sentence_transformers import SentenceTransformer, util
model_name = 'intfloat/e5-base-v2'
retriever = SentenceTransformer(model_name)

print(f"Модель '{model_name}' успешно инициализирована!")

Модель 'intfloat/e5-base-v2' успешно инициализирована!


In [1]:
!nvidia-smi

Mon Jul  6 22:17:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.211.01             Driver Version: 570.211.01     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H200                    Off |   00000000:18:00.0 Off |                    0 |
| N/A   36C    P0            205W /  700W |    4695MiB / 143771MiB |     79%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
import transformers
import torch
import random
from datasets import load_dataset
import requests
import re
import string
from collections import Counter
import json

# ==============================================================================
# 1. НАСТРОЙКА МОДЕЛЕЙ И ДАННЫХ
# ==============================================================================
print("="*50)
print("1. ИНИЦИАЛИЗАЦИЯ МОДЕЛЕЙ И ДАННЫХ")
print("="*50)

# Настройка ID модели и устройства
model_id = "PeterJinGo/SearchR1-nq_hotpotqa_train-qwen2.5-7b-em-ppo"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используемое устройство: {device}")

# Инициализация токенизатора и основной модели
print(f"Загрузка токенизатора: {model_id}...")
tokenizer = transformers.AutoTokenizer.from_pretrained(model_id)
print(f"Загрузка модели: {model_id}...")
model = transformers.AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.bfloat16, device_map="auto")
print("Основная модель успешно загружена.")

# Инициализация ретривера
from sentence_transformers import SentenceTransformer, util
retriever_model_name = 'intfloat/e5-base-v2'
print(f"Загрузка ретривера: {retriever_model_name}...")
retriever = SentenceTransformer(retriever_model_name)
print(f"Модель ретривера '{retriever_model_name}' успешно инициализирована!")


# ==============================================================================
# 2. ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ (ИНФЕРЕНС И ЭВАЛЮАЦИЯ)
# ==============================================================================
print("\n" + "="*50)
print("2. ОПРЕДЕЛЕНИЕ ВСПОМОГАТЕЛЬНЫХ ФУНКЦИЙ")
print("="*50)

# --- Функции для инференса ---

class StopOnSequence(transformers.StoppingCriteria):
    def __init__(self, target_sequences, tokenizer):
        self.target_ids = [tokenizer.encode(target_sequence, add_special_tokens=False) for target_sequence in target_sequences]
        self.target_lengths = [len(target_id) for target_id in self.target_ids]
    def __call__(self, input_ids, scores, **kwargs):
        for i, target in enumerate(self.target_ids):
            if input_ids.shape[1] >= self.target_lengths[i]:
                if torch.equal(input_ids[0, -self.target_lengths[i]:], torch.as_tensor(target, device=input_ids.device)):
                    return True
        return False

def get_query(text: str):
    pattern = re.compile(r"<search>(.*?)</search>", re.DOTALL)
    matches = pattern.findall(text)
    return matches[-1].strip() if matches else None

def search(query: str, paragraphs: list, retriever_model: SentenceTransformer, k: int = 1):
    if not query: return "No query provided."
    query_embedding = retriever_model.encode(query, convert_to_tensor=True)
    paragraph_embeddings = retriever_model.encode(paragraphs, convert_to_tensor=True)
    similarities = util.cos_sim(query_embedding, paragraph_embeddings)[0]
    top_k_indices = torch.topk(similarities, k=k).indices
    return paragraphs[top_k_indices[0].item()]

def parse_answer(text: str) -> str:
    match = re.search(r"<answer>(.*?)</answer>", text, re.DOTALL)
    if match: return match.group(1).strip()
    lines = [line.strip() for line in text.split('\n') if line.strip()]
    return lines[-1] if lines else ""
    

# --- Функции для эвалюации ---

def normalize_answer(s):
    def remove_articles(text): return re.sub(r'\b(a|an|the)\b', ' ', text)
    def white_space_fix(text): return ' '.join(text.split())
    def remove_punc(text):
        exclude = set(string.punctuation)
        return ''.join(ch for ch in text if ch not in exclude)
    def lower(text): return text.lower()
    return white_space_fix(remove_articles(remove_punc(lower(s))))

def f1_score(prediction, ground_truth):
    normalized_prediction = normalize_answer(prediction)
    normalized_ground_truth = normalize_answer(ground_truth)
    ZERO_METRIC = (0, 0, 0)
    if not normalized_prediction or not normalized_ground_truth: return ZERO_METRIC
    prediction_tokens = normalized_prediction.split()
    ground_truth_tokens = normalized_ground_truth.split()
    common = Counter(prediction_tokens) & Counter(ground_truth_tokens)
    num_same = sum(common.values())
    if num_same == 0: return ZERO_METRIC
    precision = 1.0 * num_same / len(prediction_tokens)
    recall = 1.0 * num_same / len(ground_truth_tokens)
    f1 = (2 * precision * recall) / (precision + recall)
    return f1, precision, recall

def exact_match_score(prediction, ground_truth):
    return (normalize_answer(prediction) == normalize_answer(ground_truth))

def update_answer_metrics(metrics, prediction, gold):
    em = exact_match_score(prediction, gold)
    f1, prec, recall = f1_score(prediction, gold)
    metrics['em'] += float(em)
    metrics['f1'] += f1
    metrics['prec'] += prec
    metrics['recall'] += recall

def update_sp_metrics(metrics, pred_sp_texts, gold_sp_texts):
    cur_sp_pred = set(normalize_answer(text) for text in pred_sp_texts)
    gold_sp_pred = set(normalize_answer(text) for text in gold_sp_texts)
    tp = len(cur_sp_pred.intersection(gold_sp_pred))
    fp = len(cur_sp_pred.difference(gold_sp_pred))
    fn = len(gold_sp_pred.difference(cur_sp_pred))
    prec = 1.0 * tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = 1.0 * tp / (tp + fn) if tp + fn > 0 else 0.0
    f1 = 2 * prec * recall / (prec + recall) if prec + recall > 0 else 0.0
    #em = 1.0 if fp == 0 and fn == 0 else 0.0
    #EM как в Q-rag
    em = 1.0 if fn == 0 else 0.0
    metrics['sp_em'] += em
    metrics['sp_f1'] += f1
    metrics['sp_prec'] += prec
    metrics['sp_recall'] += recall

def run_evaluation(predictions, gold_data):
    metrics = {'em': 0, 'f1': 0, 'prec': 0, 'recall': 0, 'sp_em': 0, 'sp_f1': 0, 'sp_prec': 0, 'sp_recall': 0}
    gold_map = {item['id']: item for item in gold_data}
    for sample_id, pred_answer in predictions['answer'].items():
        if sample_id not in gold_map:
            print(f"Предупреждение: ID '{sample_id}' из предсказаний не найден в эталонных данных.")
            continue
        gold_item = gold_map[sample_id]
        update_answer_metrics(metrics, pred_answer, gold_item['answer'])
        update_sp_metrics(metrics, predictions['sp'].get(sample_id, []), gold_item['supporting_facts'])
    N = len(gold_data)
    if N > 0:
        for k in metrics.keys():
            metrics[k] /= N
    print("\n" + "="*50)
    print("РЕЗУЛЬТАТЫ ЭВАЛЮАЦИИ")
    print("="*50)
    print(json.dumps(metrics, indent=4))

# ==============================================================================
# 3. ОСНОВНОЙ ЦИКЛ ИНФЕРЕНСА
# ==============================================================================
print("\n" + "="*50)
print("3. ЗАПУСК ПРОЦЕССА ИНФЕРЕНСА")
print("="*50)

# --- Настройки ---
MAX_SAMPLES = 7362
MAX_SEARCHES = 2

# --- Переменные для сбора результатов ---
predictions = {'answer': {}, 'sp': {}}
gold_data_for_eval = []

# --- Настройка критериев остановки ---
target_sequences = ["</search>", " </search>", "</search>\n", " </search>\n", "</search>\n\n", " </search>\n\n"]
stopping_criteria = transformers.StoppingCriteriaList([StopOnSequence(target_sequences, tokenizer)])
curr_eos = [151645, 151643]

# Основной цикл по сэмплам
num_samples_to_process = min(MAX_SAMPLES, len(dataset))

for i in range(num_samples_to_process):
    sample = dataset[i] # Получаем сэмпл через __getitem__
    
    sample_id = sample['id']
    question = sample['question']
    context_paragraphs = sample['chunks_texts']
    ground_truth_answer = sample['answer']
    gold_sf_indices = sample['sf_idx']
    gold_supporting_facts_texts = [context_paragraphs[j] for j in gold_sf_indices]
    
    gold_data_for_eval.append({
        'id': sample_id,
        'answer': ground_truth_answer,
        'supporting_facts': gold_supporting_facts_texts
    })
    
    print(f"\n\n################# [СЭМПЛ {i+1}/{num_samples_to_process} | ID: {sample_id}] ##################\n")
    print(f"ВОПРОС: {question}\n")
    
    search_count = 0
    retrieved_chunks_for_sample = []

    #Создаем копию списка чанков, которую можно изменять для каждого сэмпла
    available_paragraphs = list(context_paragraphs)
    
    prompt = f"""Answer the given question. You must conduct reasoning inside <think> and </think> first every time you get new information. After reasoning, if you find you lack some knowledge, you can call a search engine by <search> query </search> and it will return the top searched results between <information> and </information>. You can search as many times as you want. If you find no further external knowledge needed, you can directly provide the answer inside <answer> and </answer>.
Question: {question}\n"""
    if tokenizer.chat_template:
        prompt = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True, tokenize=False)
    
    print('---------- [Начало генерации] ----------\n')
    print(prompt, end="")
    
    final_generated_text = ""

    while search_count < MAX_SEARCHES:
        input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
        outputs = model.generate(input_ids, max_new_tokens=1024, stopping_criteria=stopping_criteria, pad_token_id=tokenizer.eos_token_id, do_sample=False)
        
        generated_tokens = outputs[0][input_ids.shape[1]:]
        output_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
        full_output_text = tokenizer.decode(generated_tokens, skip_special_tokens=False)
        
        final_generated_text += full_output_text

        if "<answer>" in output_text or outputs[0][-1].item() in curr_eos:
            print(full_output_text)
            break
            
        search_query = get_query(output_text)
        if search_query:
            print(full_output_text, end="")
            print(f' [ПОИСК {search_count + 1}/{MAX_SEARCHES}] ==> Запрос: "{search_query}"')
            search_count += 1
            search_result = search(search_query, available_paragraphs, retriever)
            retrieved_chunks_for_sample.append(search_result)
            
            #Удаляем извлеченный чанк из списка доступных, чтобы он не был найден снова
            if search_result in available_paragraphs:
                available_paragraphs.remove(search_result)

            prompt += full_output_text + f"<information>{search_result}</information>\n"
            print(f"<information>{search_result}</information>\n", end="")
        else:
            print(full_output_text)
            print("==> Модель остановилась без поискового запроса. Прерываем цикл поиска.")
            break

    if "<answer>" not in final_generated_text:
        print(f"\n==> Достигнут лимит в {MAX_SEARCHES} поисков. Запрашиваем финальный ответ...")
        force_answer_prompt = prompt + "\nYou have reached the maximum number of searches. Please provide the final answer inside <answer> and </answer> now.\n"
        input_ids = tokenizer.encode(force_answer_prompt, return_tensors='pt').to(device)
        outputs = model.generate(input_ids, max_new_tokens=1024, pad_token_id=tokenizer.eos_token_id, do_sample=False)
        
        final_answer_text = tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True)
        print(final_answer_text)
        final_generated_text += final_answer_text

    print('\n---------- [Генерация завершена] ----------')

    extracted_answer = parse_answer(final_generated_text)
    predictions['answer'][sample_id] = extracted_answer
    predictions['sp'][sample_id] = list(dict.fromkeys(retrieved_chunks_for_sample))
    
    print(f"\n> Извлеченный ответ: {extracted_answer}")
    print(f"> Эталонный ответ:  {ground_truth_answer}")

    print("\n--- Сравнение чанков (Supporting Facts) ---")
    retrieved_for_print = predictions['sp'][sample_id]
    if retrieved_for_print:
        print(f"Извлеченные чанки ({len(retrieved_for_print)} шт.):")
        for idx, chunk in enumerate(retrieved_for_print, 1):
            print(f"  {idx}. {chunk[:150]}...")
    else:
        print("Извлеченные чанки: 0 шт.")
        
    print(f"\nЭталонные чанки ({len(gold_supporting_facts_texts)} шт.):")
    for idx, chunk in enumerate(gold_supporting_facts_texts, 1):
        print(f"  {idx}. {chunk[:150]}...")

# ==============================================================================
# 4. ФИНАЛЬНАЯ ЭВАЛЮАЦИЯ
# ==============================================================================
run_evaluation(predictions, gold_data_for_eval)

1. ИНИЦИАЛИЗАЦИЯ МОДЕЛЕЙ И ДАННЫХ
Используемое устройство: cuda
Загрузка токенизатора: PeterJinGo/SearchR1-nq_hotpotqa_train-qwen2.5-7b-em-ppo...
Загрузка модели: PeterJinGo/SearchR1-nq_hotpotqa_train-qwen2.5-7b-em-ppo...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

Основная модель успешно загружена.
Загрузка ретривера: intfloat/e5-base-v2...


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Модель ретривера 'intfloat/e5-base-v2' успешно инициализирована!

2. ОПРЕДЕЛЕНИЕ ВСПОМОГАТЕЛЬНЫХ ФУНКЦИЙ

3. ЗАПУСК ПРОЦЕССА ИНФЕРЕНСА


################# [СЭМПЛ 1/1000 | ID: 5a7613c15542994ccc9186bf] ##################

ВОПРОС: VIVA Media AG changed it's name in 2004. What does their new acronym stand for

---------- [Начало генерации] ----------

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Answer the given question. You must conduct reasoning inside <think> and </think> first every time you get new information. After reasoning, if you find you lack some knowledge, you can call a search engine by <search> query </search> and it will return the top searched results between <information> and </information>. You can search as many times as you want. If you find no further external knowledge needed, you can directly provide the answer inside <answer> and </answer>.
Question: VIVA Media AG changed it's name in 2004. What does their new acronym stand for
<|im_

KeyboardInterrupt: 